# Phase 2: Data Preprocessing & Feature Engineering
### IT24102884 — Shankar.M

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

warnings.filterwarnings('ignore')

C:\Users\MAHEN\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Load dataset
df = pd.read_csv('data/bank-full.csv', sep=';')
print('Dataset shape:', df.shape)

Dataset shape: (45211, 17)


## 2.1 Handle Data Leakage — Drop Duration

In [3]:
# Rationale: 'duration' is only known during or after the call.
# Including it creates an unrealistic model that cannot predict beforehand.
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])
print('Dropped duration column. New shape:', df.shape)

Dropped duration column. New shape: (45211, 16)


## 2.2 Handle Unknown Values

In [4]:
# For job, education, contact, and poutcome, we keep 'unknown' as its own category.
# - job (0.64%): small proportion, but may carry signal.
# - education (4.11%): reasonable proportion, missingness may have meaning.
# - contact (28.80%): too many to drop rows.
# - poutcome (81.75%): 'unknown' means they were never contacted before.
unknown_cols = ['job', 'education', 'contact', 'poutcome']
for col in unknown_cols:
    print(f"{col} 'unknown' count:", (df[col] == 'unknown').sum())
print('\nDecision: Retaining unknown as a discrete category for all columns.')

job 'unknown' count: 288
education 'unknown' count: 1857
contact 'unknown' count: 13020
poutcome 'unknown' count: 36959

Decision: Retaining unknown as a discrete category for all columns.


## 2.3 Feature Engineering — pdays Sentinel Value

In [5]:
# Then we replace -1 with 0 in pdays so the numeric value doesn't skew models negatively.
df['previously_contacted'] = (df['pdays'] >= 0).astype(int)
df.loc[df['pdays'] == -1, 'pdays'] = 0

print('previously_contacted distribution:')
print(df['previously_contacted'].value_counts())
print('\nRationale: A binary flag clearly captures the "never contacted" state, allowing pdays to represent actual days for those contacted.')

previously_contacted distribution:
previously_contacted
0    36954
1     8257
Name: count, dtype: int64

Rationale: A binary flag clearly captures the "never contacted" state, allowing pdays to represent actual days for those contacted.


## 2.4 Encode Target Variable

In [6]:
df['y'] = df['y'].map({'yes': 1, 'no': 0})
print('Target distribution after mapping:')
print(df['y'].value_counts(normalize=True) * 100)

Target distribution after mapping:
y
0    88.30152
1    11.69848
Name: proportion, dtype: float64


## 2.5 Encode Binary Categorical Features

In [7]:
binary_cols = ['default', 'housing', 'loan']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})
print('Encoded binary features:', binary_cols)

Encoded binary features: ['default', 'housing', 'loan']


## 2.6 Encode Multi-Class Categorical Features (One-Hot Encoding)

In [8]:
multi_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True, dtype=int)
print('New shape after one-hot encoding:', df.shape)
print('\nNew columns created:', [c for c in df.columns if any(c.startswith(mc) for mc in multi_cols)])

New shape after one-hot encoding: (45211, 43)

New columns created: ['job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'job_unknown', 'marital_married', 'marital_single', 'education_secondary', 'education_tertiary', 'education_unknown', 'contact_telephone', 'contact_unknown', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'poutcome_other', 'poutcome_success', 'poutcome_unknown']


## 2.7 Handle Outliers

In [9]:
outlier_cols = ['balance', 'campaign', 'previous']

for col in outlier_cols:
    print(f"{col} percentiles (1st, 5th, 95th, 99th):")
    print(np.percentile(df[col], [1, 5, 95, 99]))

# IQR-based capping at 1st and 99th percentile for balance
bal_1 = np.percentile(df['balance'], 1)
bal_99 = np.percentile(df['balance'], 99)
df['balance'] = np.clip(df['balance'], bal_1, bal_99)

# Cap campaign and previous at 99th percentile
camp_99 = np.percentile(df['campaign'], 99)
df['campaign'] = np.clip(df['campaign'], None, camp_99)

prev_99 = np.percentile(df['previous'], 99)
df['previous'] = np.clip(df['previous'], None, prev_99)

# Comment: Tree-based models handle outliers well, but capping helps models like Logistic Regression.

balance percentiles (1st, 5th, 95th, 99th):
[ -627.   -172.   5768.  13164.9]
campaign percentiles (1st, 5th, 95th, 99th):
[ 1.  1.  8. 16.]
previous percentiles (1st, 5th, 95th, 99th):
[0.  0.  3.  8.9]


## 2.8 Train-Test Split

In [10]:
X = df.drop(columns=['y'])
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Train shapes:', X_train.shape, y_train.shape)
print('Test shapes:', X_test.shape, y_test.shape)
print('\nTrain class distribution:\n', y_train.value_counts(normalize=True))
print('\nTest class distribution:\n', y_test.value_counts(normalize=True))

Train shapes: (36168, 42) (36168,)
Test shapes: (9043, 42) (9043,)

Train class distribution:
 y
0    0.883018
1    0.116982
Name: proportion, dtype: float64

Test class distribution:
 y
0    0.883003
1    0.116997
Name: proportion, dtype: float64


## 2.9 Feature Scaling

In [11]:
# RobustScaler uses median and IQR, making it robust to outliers.
num_cols = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
scaler = RobustScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Applied RobustScaler to:', num_cols)

Applied RobustScaler to: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']


## 2.10 Handle Class Imbalance — SMOTE

In [12]:
print('Class distribution BEFORE SMOTE:\n', y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print('\nClass distribution AFTER SMOTE:\n', y_train_resampled.value_counts())
# Comment: SMOTE is ONLY applied to training data to prevent data leakage.

Class distribution BEFORE SMOTE:
 y
0    31937
1     4231
Name: count, dtype: int64

Class distribution AFTER SMOTE:
 y
0    31937
1    31937
Name: count, dtype: int64


## 2.11 Save Preprocessed Data

In [13]:
os.makedirs('preprocessed', exist_ok=True)

X_train_resampled.to_csv('preprocessed/X_train.csv', index=False)
y_train_resampled.to_csv('preprocessed/y_train.csv', index=False)
X_test.to_csv('preprocessed/X_test.csv', index=False)
y_test.to_csv('preprocessed/y_test.csv', index=False)

joblib.dump(scaler, 'preprocessed/scaler.pkl')

print('Saved preprocessed datasets and scaler to preprocessed/ directory.')

Saved preprocessed datasets and scaler to preprocessed/ directory.


## 2.12 Preprocessing Decision Log

| Decision | Choice Made | Rationale | Alternatives Considered |
|---|---|---|---|
| Duration Drop | Dropped 'duration' column | Duration is only known after call ends (data leakage) | Keep it (unrealistic model) |
| Unknown Handling | Retained as 'unknown' category | Unknowns may represent specific patterns | Imputation or row dropping |
| pdays Engineering | Binary flag 'previously_contacted' | Clearly separates contacted vs. not contacted | Leave -1 (distorts continuous feature) |
| Encoding Strategy | Map binary / One-Hot for multi | Standard practice for distinct categories | Ordinal encoding |
| Outlier Capping | 1st/99th percentile IQR caps | Extreme values can skew linear models | Drop rows (lose data) |
| Scaler Choice | RobustScaler | Median/IQR approach resists extreme outliers better | StandardScaler (skewed by outliers) |
| Imbalance Handling | SMOTE on Train Data | Synthetic minority generation prevents bias towards majority | Undersampling (lose data) |